# Observation Windows — Activity Features

Build **activity features** per customer and time window from the cleaned Online Retail II dataset.

Source: `data/processed/online_retail_II_cleaned.csv`

## 1. Load & prepare

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = Path("../data/processed/online_retail_II_cleaned.csv")
OUTPUT_DIR = Path("../data/processed")

df = pd.read_csv(DATA_PATH, parse_dates=["InvoiceDate"])
df = df.rename(columns={"Customer ID": "CustomerID"})
df["CustomerID"] = df["CustomerID"].astype("int64")
df["TotalPrice"] = df["Quantity"] * df["Price"]
df["purchase_date"] = df["InvoiceDate"].dt.normalize()

print("Shape:", df.shape)
print("Date range:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
print("Unique customers:", df["CustomerID"].nunique())
df.head()

Shape: (776844, 11)
Date range: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00
Unique customers: 5853


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,Month,TotalPrice,purchase_date
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009-12,83.4,2009-12-01
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-12,81.0,2009-12-01
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-12,81.0,2009-12-01
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009-12,100.8,2009-12-01
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-12,30.0,2009-12-01


## 2. Activity-feature builder

For each `(CustomerID, window_id)` compute:

- `orders` — number of invoices
- `spend` — total spend (`Quantity * Price`)
- `totalQuntaity` — total quantity
- `avargeOrderValue` — average order value
- `unique_products` — distinct stock codes
- `active_days` — distinct purchase dates
- `items_per_order` — line items per invoice (approx)

In [2]:
def build_activity_features(df: pd.DataFrame, window_days: int = 30) -> pd.DataFrame:
    """Aggregate purchase activity per customer and fixed-length observation window."""
    data = df.copy()
    start = data["InvoiceDate"].min().normalize()

    days_from_start = (data["InvoiceDate"] - start).dt.days
    data["window_id"] = days_from_start // window_days
    data["window_start"] = start + pd.to_timedelta(data["window_id"] * window_days, unit="D")
    data["window_end"] = data["window_start"] + pd.Timedelta(days=window_days)

    activity = (
        data.groupby(["CustomerID", "window_id", "window_start", "window_end"], as_index=False)
        .agg(
            orders=("Invoice", "nunique"),
            spend=("TotalPrice", "sum"),
            totalQuantity=("Quantity", "sum"),
            unique_products=("StockCode", "nunique"),
            active_days=("purchase_date", "nunique"),
            line_items=("Invoice", "size"),
            first_purchase=("InvoiceDate", "min"),
            last_purchase=("InvoiceDate", "max"),
        )
    )

    activity["avargeOrderValue"] = activity["spend"] / activity["orders"]
    activity["items_per_order"] = activity["line_items"] / activity["orders"]
    activity["window_days"] = window_days

    activity = activity.sort_values(["CustomerID", "window_id"]).reset_index(drop=True)
    return activity

## 3. Build activity features for 30 / 60 / 90 day windows

In [3]:
WINDOWS = [30, 60, 90]
activity_by_window = {}

for w in WINDOWS:
    activity = build_activity_features(df, window_days=w)
    activity_by_window[w] = activity
    print(
        f"{w:>3}-day window -> rows: {len(activity):,} | "
        f"customers: {activity['CustomerID'].nunique():,} | "
        f"windows: {activity['window_id'].nunique()}"
    )

 30-day window -> rows: 25,537 | customers: 5,853 | windows: 25
 60-day window -> rows: 20,937 | customers: 5,853 | windows: 13
 90-day window -> rows: 18,015 | customers: 5,853 | windows: 9


## 4. Preview (default: 30-day)

In [4]:
activity_30 = activity_by_window[30]
display_cols = [
    "CustomerID", "window_id", "window_start", "window_end",
    "orders", "spend", "totalQuantity","avargeOrderValue",
    "unique_products", "active_days", "items_per_order",
]

activity_30[display_cols].head(10)

,CustomerID,window_id,window_start,window_end,orders,spend,totalQuantity,avargeOrderValue,unique_products,active_days,items_per_order
0,12346,3,2010-03-01,2010-03-31,1,27.05,5,27.05,5,1,5.0
1,12346,6,2010-05-30,2010-06-29,1,142.31,19,142.31,19,1,19.0
2,12346,13,2010-12-26,2011-01-25,1,77183.60,74215,77183.60,1,1,1.0
3,12347,11,2010-10-27,2010-11-26,1,611.53,509,611.53,40,1,40.0
4,12347,12,2010-11-26,2010-12-26,1,711.79,319,711.79,31,1,31.0
5,12347,14,2011-01-25,2011-02-24,1,475.39,315,475.39,29,1,29.0
6,12347,16,2011-03-26,2011-04-25,1,636.25,483,636.25,24,1,24.0
7,12347,18,2011-05-25,2011-06-24,1,382.52,196,382.52,18,1,18.0
8,12347,20,2011-07-24,2011-08-23,1,584.91,277,584.91,22,1,22.0
9,12347,23,2011-10-22,2011-11-21,1,1294.32,676,1294.32,47,1,47.0


In [5]:
activity_30[display_cols].describe()

,CustomerID,window_id,window_start,window_end,orders,spend,totalQuantity,avargeOrderValue,unique_products,active_days,items_per_order
count,25537.000000,25537.000000,25537,25537,25537.000000,25537.000000,25537.000000,25537.000000,25537.000000,25537.000000,25537.000000
mean,15304.739163,12.613933,2010-12-14 10:01:53.623370,2011-01-13 10:01:53.623370,1.433215,668.877657,411.129107,430.576984,28.163762,1.287622,22.159108
min,12346.000000,0.000000,2009-12-01 00:00:00,2009-12-31 00:00:00,1.000000,0.850000,1.000000,0.850000,1.000000,1.000000,1.000000
25%,13837.000000,7.000000,2010-06-29 00:00:00,2010-07-29 00:00:00,1.000000,207.240000,102.000000,178.753333,10.000000,1.000000,9.000000
50%,15280.000000,12.000000,2010-11-26 00:00:00,2010-12-26 00:00:00,1.000000,344.610000,196.000000,304.950000,19.000000,1.000000,16.500000
75%,16781.000000,19.000000,2011-06-24 00:00:00,2011-07-24 00:00:00,1.000000,610.690000,365.000000,460.410000,35.000000,1.000000,28.000000
max,18287.000000,24.000000,2011-11-21 00:00:00,2011-12-21 00:00:00,39.000000,168469.600000,93230.000000,168469.600000,801.000000,18.000000,500.333333
std,1708.139693,7.243144,NaN,NaN,1.276602,2114.443047,1673.964492,1309.157401,33.043281,0.835939,21.316925


## 5. Example: one customer across windows

In [7]:
# Pick a customer with activity in multiple windows
multi_window_customers = (
    activity_30.groupby("CustomerID")["window_id"]
    .nunique()
    .sort_values(ascending=False)
)

sample_customer = multi_window_customers.index[0]
print("Sample CustomerID:", sample_customer, "| active windows:", multi_window_customers.iloc[0])

activity_30.loc[
    activity_30["CustomerID"] == sample_customer,
    display_cols,
].head(15)



len(multi_window_customers)

Sample CustomerID: 14606 | active windows: 25


5853

## Build Previous Features 

In [2]:
# Load activity features 30 day 
feature_df=pd.read_csv("../data/processed/activity_features_30d.csv")
feature_df.head(10)

,CustomerID,window_id,window_start,window_end,orders,spend,totalQuantity,unique_products,active_days,line_items,first_purchase,last_purchase,avargeOrderValue,items_per_order,window_days
0,12346,3,2010-03-01,2010-03-31,1,27.05,5,5,1,5,2010-03-02 13:08:00,2010-03-02 13:08:00,27.05,5.0,30
1,12346,6,2010-05-30,2010-06-29,1,142.31,19,19,1,19,2010-06-28 13:53:00,2010-06-28 13:53:00,142.31,19.0,30
2,12346,13,2010-12-26,2011-01-25,1,77183.60,74215,1,1,1,2011-01-18 10:01:00,2011-01-18 10:01:00,77183.60,1.0,30
3,12347,11,2010-10-27,2010-11-26,1,611.53,509,40,1,40,2010-10-31 14:20:00,2010-10-31 14:20:00,611.53,40.0,30
4,12347,12,2010-11-26,2010-12-26,1,711.79,319,31,1,31,2010-12-07 14:57:00,2010-12-07 14:57:00,711.79,31.0,30
5,12347,14,2011-01-25,2011-02-24,1,475.39,315,29,1,29,2011-01-26 14:30:00,2011-01-26 14:30:00,475.39,29.0,30
6,12347,16,2011-03-26,2011-04-25,1,636.25,483,24,1,24,2011-04-07 10:43:00,2011-04-07 10:43:00,636.25,24.0,30
7,12347,18,2011-05-25,2011-06-24,1,382.52,196,18,1,18,2011-06-09 13:01:00,2011-06-09 13:01:00,382.52,18.0,30
8,12347,20,2011-07-24,2011-08-23,1,584.91,277,22,1,22,2011-08-02 08:48:00,2011-08-02 08:48:00,584.91,22.0,30
9,12347,23,2011-10-22,2011-11-21,1,1294.32,676,47,1,47,2011-10-31 12:25:00,2011-10-31 12:25:00,1294.32,47.0,30


In [4]:
previous_features = [
    "orders",
    "spend",
    "totalQuantity",
    "avargeOrderValue",
    "unique_products",
    "active_days",
    "items_per_order"
]

for col in previous_features:
    feature_df[f"prev_{col}"] = (
        feature_df
        .sort_values(["CustomerID", "window_id"])
        .groupby("CustomerID")[col]
        .shift(1)
    )
feature_df.head()

,CustomerID,window_id,window_start,window_end,orders,spend,totalQuantity,unique_products,active_days,line_items,...,avargeOrderValue,items_per_order,window_days,prev_orders,prev_spend,prev_totalQuantity,prev_avargeOrderValue,prev_unique_products,prev_active_days,prev_items_per_order
0,12346,3,2010-03-01,2010-03-31,1,27.05,5,5,1,5,...,27.05,5.0,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12346,6,2010-05-30,2010-06-29,1,142.31,19,19,1,19,...,142.31,19.0,30,1.0,27.05,5.0,27.05,5.0,1.0,5.0
2,12346,13,2010-12-26,2011-01-25,1,77183.60,74215,1,1,1,...,77183.60,1.0,30,1.0,142.31,19.0,142.31,19.0,1.0,19.0
3,12347,11,2010-10-27,2010-11-26,1,611.53,509,40,1,40,...,611.53,40.0,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12347,12,2010-11-26,2010-12-26,1,711.79,319,31,1,31,...,711.79,31.0,30,1.0,611.53,509.0,611.53,40.0,1.0,40.0


## Change Features 

In [7]:
# Change Features: compare current window with previous window

change_features = [
    "orders",
    "spend",
    "totalQuantity",
    "avargeOrderValue",
    "unique_products",
    "active_days",
    "items_per_order"
]

for col in change_features:
    feature_df[f"{col}_change_pct"] = (
        (feature_df[col] - feature_df[f"prev_{col}"])
        / feature_df[f"prev_{col}"]
    )

# Replace infinite values caused by division by zero with NaN
feature_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# First window of each customer has no previous window
# so it cannot have a change value
feature_df = feature_df.dropna(
    subset=[f"{col}_change_pct" for col in change_features]
)

feature_df.head()

,CustomerID,window_id,window_start,window_end,orders,spend,totalQuantity,unique_products,active_days,line_items,...,prev_unique_products,prev_active_days,prev_items_per_order,orders_change_pct,spend_change_pct,totalQuantity_change_pct,avargeOrderValue_change_pct,unique_products_change_pct,active_days_change_pct,items_per_order_change_pct
1,12346,6,2010-05-30,2010-06-29,1,142.31,19,19,1,19,...,5.0,1.0,5.0,0.0,4.260998,2.800000,4.260998,2.800000,0.0,2.800000
2,12346,13,2010-12-26,2011-01-25,1,77183.60,74215,1,1,1,...,19.0,1.0,19.0,0.0,541.362448,3905.052632,541.362448,-0.947368,0.0,-0.947368
4,12347,12,2010-11-26,2010-12-26,1,711.79,319,31,1,31,...,40.0,1.0,40.0,0.0,0.163949,-0.373281,0.163949,-0.225000,0.0,-0.225000
5,12347,14,2011-01-25,2011-02-24,1,475.39,315,29,1,29,...,31.0,1.0,31.0,0.0,-0.332120,-0.012539,-0.332120,-0.064516,0.0,-0.064516
6,12347,16,2011-03-26,2011-04-25,1,636.25,483,24,1,24,...,29.0,1.0,29.0,0.0,0.338375,0.533333,0.338375,-0.172414,0.0,-0.172414


In [10]:

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_DIR / f"online_retail_ll_features.csv"
feature_df.to_csv(out_path, index=False)
print(f"Saved: {out_path.resolve()} | shape={feature_df.shape}")


Saved: F:\GSG26_Course\fina_ml_project\data\processed\online_retail_ll_features.csv | shape=(19684, 29)


## 6. Save activity-feature tables

In [9]:

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for w, activity in activity_by_window.items():
    out_path = OUTPUT_DIR / f"activity_features_{w}d.csv"
    activity.to_csv(out_path, index=False)
    print(f"Saved: {out_path.resolve()} | shape={activity.shape}")


Saved: F:\GSG26_Course\fina_ml_project\data\processed\activity_features_30d.csv | shape=(25537, 15)
Saved: F:\GSG26_Course\fina_ml_project\data\processed\activity_features_60d.csv | shape=(20937, 15)
Saved: F:\GSG26_Course\fina_ml_project\data\processed\activity_features_90d.csv | shape=(18015, 15)
